# Session 2 · Part 3 — Visualize predicted proteins

**Independent checkpoint:** load predictions from the Session 2 checkpoint when available, otherwise use the committed table, then create spatial maps.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import matplotlib.pyplot as plt

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.plotting import plot_spatial_feature

dataset = load_tutorial_data(paths.raw_data, allow_demo=False)
spots = dataset.spots
prediction_path = preferred_prediction_path(paths)
predicted_proteins = load_prediction_table(str(prediction_path))
common_spots = spots.index.intersection(predicted_proteins.index)
if common_spots.empty:
    raise ValueError("Spatial data and predictions have no shared spot IDs; check that the assets match.")
spots = spots.loc[common_spots]
predicted_proteins = predicted_proteins.loc[common_spots]


In [ ]:
proteins_to_plot = list(predicted_proteins.columns[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(spots, predicted_proteins[protein], f"Predicted {protein}", ax=ax)
plt.tight_layout()
figure_path = paths.figures / "session02_predicted_proteins.png"
plt.savefig(figure_path, dpi=160)
plt.show()

manifest = write_checkpoint(
    "2.3", [prediction_path, figure_path],
    summary={"spots": len(common_spots), "proteins_plotted": proteins_to_plot}, start=paths.root
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

Session 2 is complete when the prediction provenance and spatial map are both available.